# E3/E4 — ANOVA ROC curves (`fig:roc_effect`, `fig:roc_sample`)

Reproduce the paper figures for the one-way ANOVA hypothesis test on $\hat{\sigma}$:

1. **Effect size** (fixed $n=500$, $\sigma_1=-1$, sweep $\sigma_2 \in \{-1,-1.5,-2,-2.5\}$)
2. **Sample size** (fixed $\sigma_1=-1$, $\sigma_2=-1.5$, sweep $n \in \{10,100,500,1000,2000\}$)

Both use $d \in \{0,1,2\}$. For each Monte Carlo replicate we simulate two groups of LG graphs, estimate $\hat{\sigma}$ on each graph, run `scipy.stats.f_oneway`, and build ROC curves from the resulting $p$-values.

Set `MODE = "SMOKE" | "INSIGHT" | "DEV" | "PAPER"`. Cached CSV: `images/correction_paper/roc_sweeps_<hash>.csv`.

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

from pathlib import Path
import time
import pandas as pd

from logit_graph.experiments import (
    PRESETS,
    run_roc_sweeps,
    plot_roc_effect_size,
    plot_roc_sample_size,
)

MODE = "PAPER"  # SMOKE | INSIGHT | DEV | PAPER
USE_CACHE = True
N_JOBS = 2  # keep low: each job holds an n=500 graph in memory

OUT_DIR = (Path("..") / ".." / "images" / "correction_paper").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = PRESETS[MODE]["roc"]
print(
    f"MODE={MODE}, n_effect={cfg.n_effect}, n_values={cfg.n_values}, "
    f"d={cfg.d_values}, reps={cfg.n_reps}, exps={cfg.n_experiments}, "
    f"iter_cap={cfg.iter_cap}"
)

In [ ]:
t0 = time.perf_counter()
effect_df, sample_df = run_roc_sweeps(cfg, OUT_DIR, use_cache=USE_CACHE, n_jobs=N_JOBS)
combined = pd.concat([effect_df, sample_df], ignore_index=True)
print(f"Sweep finished in {time.perf_counter() - t0:.1f}s")

plot_roc_effect_size(
    combined,
    OUT_DIR / "roc_effect_size.png",
    sigma1=cfg.sigma1,
    n_fixed=cfg.n_effect,
)
plot_roc_sample_size(
    combined,
    OUT_DIR / "roc_sample_size.png",
    sigma1=cfg.sigma1,
    sigma2=cfg.sigma2_fixed,
)
print("Saved", OUT_DIR / "roc_effect_size.png")
print("Saved", OUT_DIR / "roc_sample_size.png")

In [ ]:
effect_summary = (
    effect_df.groupby(["d", "sigma2"])["power_at_005"]
    .first()
    .reset_index()
    .rename(columns={"power_at_005": "reject_at_0.05"})
)
sample_summary = (
    sample_df.groupby(["d", "n"])["power_at_005"]
    .first()
    .reset_index()
    .rename(columns={"power_at_005": "reject_at_0.05"})
)

print("Effect-size sweep — rejection rate at alpha=0.05:")
display(effect_summary)
print("\nSample-size sweep — rejection rate at alpha=0.05:")
display(sample_summary)

## Expected qualitative behaviour (paper Section 4.1.2)

- **Null** ($\sigma_2 = \sigma_1$): ROC should lie near the diagonal (Type I calibration).
- **Effect size**: curves should lift above the diagonal as $|\sigma_1 - \sigma_2|$ grows.
- **Sample size**: power should increase with $n$; small $n$ (e.g. 10) stays near the diagonal.

The legacy figure in `notebooks/changes_article/fig2_roc_curves.ipynb` used a synthetic
$\hat{\sigma} \sim \mathcal{N}(\sigma, \mathrm{se}(n,d))$ engine. This notebook runs the
**real** LG simulation + offset logistic estimator pipeline (`simulate_graph`, `estimate_sigma_from_graph`).

Use `MODE="PAPER"` for the full grid; expect long runtimes when $n \gtrsim 500$ and $d \geq 1$
because each graph requires Layer-2 Gibbs mixing.